## 01 Setup

In [4]:
import os, json, time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")               # the notebook lives in notebooks/, .env in the root
KEY = os.getenv("CENSUS_KEY")
BASE = ".."                          # project root, seen from notebooks/
URL = "https://api.census.gov/data/timeseries/intltrade/imports/hs"

print("Key loaded:", KEY is not None)

Key loaded: True


## 02 test and country codes

In [5]:
params = {"get": "CTY_CODE,CTY_NAME,GEN_VAL_MO", "YEAR": "2024", "MONTH": "01", "key": KEY}
r = requests.get(URL, params=params, timeout=120)
print("Status:", r.status_code)

data = r.json()
countries = pd.DataFrame(data[1:], columns=data[0])
countries[countries["CTY_NAME"].str.contains("MEXICO|CHINA|TAIWAN|TOTAL FOR ALL")]

Status: 200


,CTY_CODE,CTY_NAME,GEN_VAL_MO,YEAR,MONTH
0,-,TOTAL FOR ALL COUNTRIES,253526879905,2024,01
14,2010,MEXICO,37826106079,2024,01
158,5700,CHINA,35770786274,2024,01
162,5830,TAIWAN,8195028928,2024,01


## 03 partner codes, taken from the test table

In [6]:
codes = dict(zip(countries["CTY_NAME"], countries["CTY_CODE"]))
PARTNERS = {"MX": codes["MEXICO"], "CN": codes["CHINA"], "TW": codes["TAIWAN"], "WORLD": None}
PARTNERS

{'MX': '2010', 'CN': '5700', 'TW': '5830', 'WORLD': None}

## 04 HS2 catalog with sectors

In [7]:
params = {"get": "I_COMMODITY,I_COMMODITY_SDESC,GEN_VAL_MO",
          "YEAR": "2024", "MONTH": "01", "COMM_LVL": "HS2", "key": KEY}
d = requests.get(URL, params=params, timeout=120).json()
hs2 = pd.DataFrame(d[1:], columns=d[0])[["I_COMMODITY", "I_COMMODITY_SDESC"]]
hs2.columns = ["hs2", "hs2_desc"]

def sector(ch):
    ch = int(ch)
    if ch <= 24: return "Agri-food"
    if ch <= 27: return "Minerals & fuels"
    if ch <= 40: return "Chemicals & plastics"
    if ch <= 49: return "Leather, wood & paper"
    if ch <= 67: return "Textiles, apparel & footwear"
    if ch <= 71: return "Stone, glass & precious metals"
    if ch <= 83: return "Base metals"
    if ch == 84: return "Machinery"
    if ch == 85: return "Electrical & electronics"
    if ch == 87: return "Vehicles & parts"
    if ch <= 89: return "Other transport"
    if ch <= 92: return "Instruments"
    return "Other"

hs2["sector"] = hs2["hs2"].apply(sector)
hs2.to_csv(f"{BASE}/data/reference/sector_map.csv", index=False)
hs2["sector"].value_counts()

sector
Agri-food                         24
Textiles, apparel & footwear      18
Chemicals & plastics              13
Base metals                       11
Leather, wood & paper              9
Other                              7
Stone, glass & precious metals     4
Minerals & fuels                   3
Other transport                    3
Instruments                        3
Machinery                          1
Electrical & electronics           1
Vehicles & parts                   1
Name: count, dtype: int64

## 05 full pull

In [13]:
RAW = f"{BASE}/data/raw"
os.makedirs(RAW, exist_ok=True)
YEARS = range(2017, 2026)

def fetch(params, retries=3):
    for i in range(retries):
        r = requests.get(URL, params=params, timeout=180)
        if r.status_code == 200: return r.json()
        if r.status_code == 204: return []          # valid request, no data
        time.sleep(10 * (i + 1))
    raise RuntimeError(f"{r.status_code}: {r.text[:200]}")

for name, code in PARTNERS.items():
    for y in YEARS:
        out = f"{RAW}/imports_hs4_{name}_{y}.json"
        if os.path.exists(out):
            continue                                  # resume safely if it stops
        params = {"get": "I_COMMODITY,I_COMMODITY_SDESC,GEN_VAL_MO,MONTH",
                  "YEAR": str(y), "COMM_LVL": "HS4", "key": KEY}
        if code:                                      # WORLD = no filter → total
            params["get"] = "CTY_CODE,CTY_NAME," + params["get"]
            params["CTY_CODE"] = code
        data = fetch(params)
        with open(out, "w") as f:
            json.dump(data, f)
        print(name, y, max(len(data) - 1, 0), "rows")
        time.sleep(1)

## 06 Santy check

In [14]:
import glob
frames = []
for path in glob.glob(f"{RAW}/*.json"):
    data = json.load(open(path))
    if not data:
        continue
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.loc[:, ~df.columns.duplicated()]          # drop repeated columns (CTY_CODE)
    _, _, partner, year = os.path.basename(path)[:-5].split("_")
    df["partner"], df["YEAR"] = partner, int(year)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw["GEN_VAL_MO"] = pd.to_numeric(raw["GEN_VAL_MO"])
print("Columns:", list(raw.columns))
print("Total rows:", len(raw))

Columns: ['I_COMMODITY', 'I_COMMODITY_SDESC', 'GEN_VAL_MO', 'MONTH', 'YEAR', 'COMM_LVL', 'partner', 'CTY_CODE', 'CTY_NAME']
Total rows: 462984


## 07 Compact Chech

In [15]:
files = glob.glob(f"{RAW}/*.json")
months = raw.groupby(["partner", "YEAR"])["MONTH"].nunique()
totals = (raw.groupby(["partner", "YEAR"])["GEN_VAL_MO"].sum() / 1e9).round(1)

print("Files:", len(files))
print("Total rows:", len(raw))
print("Partner-years with fewer than 12 months:", (months < 12).sum())
print("Mexico 2024 (USD bn):", totals.loc[("MX", 2024)])
print("WORLD 2024 (USD bn):", totals.loc[("WORLD", 2024)])
print("Share of MX, CN, TW in WORLD 2024:",
      (totals.loc[[("MX", 2024), ("CN", 2024), ("TW", 2024)]] / totals.loc[("WORLD", 2024)]).round(3).tolist())

Files: 36
Total rows: 462984
Partner-years with fewer than 12 months: 0
Mexico 2024 (USD bn): 503.1
WORLD 2024 (USD bn): 3263.9
Share of MX, CN, TW in WORLD 2024: [0.154, 0.135, 0.036]
